# MBG Inference — Context Filter
Standalone notebook to run relevance filtering on any CSV using the fine-tuned IndoBERT model.

**Use this when you want to filter a new batch of tweets without running the full pipeline.**

Outputs: `tweets_relevant.csv`, `tweets_rejected.csv`, `tweets_borderline.csv`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, time

os.environ["RUNTIME_MODE"] = "colab"

DRIVE_BASE = "/content/drive/MyDrive/mbg"
CODE_DIR   = "/content/mbg-pipeline"
GITHUB_REPO = "https://github.com/FatwaArya/mbg-analysis"

# Clone/pull repo
if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", GITHUB_REPO, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull"], check=True)

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print("✅ Ready")

In [ ]:
import torch
assert torch.cuda.is_available(), "❌ No GPU — Runtime → Change runtime type → T4 GPU"
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
subprocess.run(["pip", "install", "-q", "-r", f"{CODE_DIR}/requirements.txt"], check=True)
print("✅ Dependencies installed")

In [ ]:
# ── Verify model exists on Drive ──────────────────────────────────────────────
MODEL_DIR = f"{DRIVE_BASE}/model"

if not os.path.exists(f"{MODEL_DIR}/config.json"):
    print("Downloading model from Spaces (~475MB)...")
    subprocess.run(["pip", "install", "-q", "s3cmd"], check=True)
    os.makedirs(MODEL_DIR, exist_ok=True)
    subprocess.run([
        "s3cmd", "get", "--recursive",
        "s3://mbg-scraper-network-20260419071440/models/mbg-indobert-finetuned/",
        MODEL_DIR + "/"
    ], check=True)

print(f"✅ Model ready at {MODEL_DIR}")

In [ ]:
# ── Point to your input CSV ───────────────────────────────────────────────────
# Change this to any CSV with a 'text' column
INPUT_CSV = f"{DRIVE_BASE}/data/raw/mbg-corpus.csv"

import pandas as pd
df_raw = pd.read_csv(INPUT_CSV, low_memory=False)
print(f"✅ Loaded: {len(df_raw):,} rows from {INPUT_CSV}")
df_raw.head(2)

In [ ]:
# ── Run inference via inference.py (uses runtime.py for paths/device) ─────────
# Copy input to the expected raw dir so inference.py can find it
RAW_DIR    = f"{DRIVE_BASE}/data/raw"
OUTPUT_DIR = f"{DRIVE_BASE}/data/output"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# If INPUT_CSV is not already in RAW_DIR, copy it there
import shutil
if os.path.dirname(os.path.abspath(INPUT_CSV)) != os.path.abspath(RAW_DIR):
    shutil.copy(INPUT_CSV, f"{RAW_DIR}/{os.path.basename(INPUT_CSV)}")
    print(f"Copied input to {RAW_DIR}/")

print("Running inference.py...")
t0 = time.time()
result = !RUNTIME_MODE=colab python3 {CODE_DIR}/inference.py 2>&1
for line in result:
    print(line)
print(f"\nDone in {(time.time()-t0)/60:.1f} min")

In [ ]:
# ── Results summary ───────────────────────────────────────────────────────────
relevant   = pd.read_csv(f"{OUTPUT_DIR}/tweets_relevant.csv")
rejected   = pd.read_csv(f"{OUTPUT_DIR}/tweets_rejected.csv")
borderline = pd.read_csv(f"{OUTPUT_DIR}/tweets_borderline.csv")

total = len(relevant) + len(rejected)
print(f"RELEVANT   : {len(relevant):,} ({len(relevant)/total*100:.1f}%)")
print(f"REJECTED   : {len(rejected):,} ({len(rejected)/total*100:.1f}%)")
print(f"BORDERLINE : {len(borderline):,} (confidence < 0.80)")
print(f"\nSaved to: {OUTPUT_DIR}/")
relevant[["text", "predicted_confidence"]].sample(5)